# Flashcard Agent Test (Stage 11)

Full pipeline:
1. Load ChromaDB index (retriever)
2. Retrieve relevant chunks for a topic
3. Generate study flashcards via `FlashcardAgent`

Each flashcard includes:
- **front** — term or question
- **back** — answer or explanation

**Prerequisites:**
- Index already created (`01_test_rag_pipeline.ipynb`, Part A)
- `.env` with `HUGGINGFACE_API_TOKEN` (Stage 7)
- `uv sync`

## Configuration

In [2]:
import json
import os
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VECTOR_DB_PATH = ROOT / "data/vector_db"

TOP_K = 5
QUESTION = "Key concepts for studying the Transformer architecture"

token = os.getenv("HUGGINGFACE_API_TOKEN") or os.getenv("HF_TOKEN")

print(f"Project root: {ROOT}")
print(f"Vector DB: {VECTOR_DB_PATH}")
print(f"Default topic: {QUESTION}")
print(f"HF token: {'configured' if token else 'NOT configured'}")

Project root: d:\Github\research-workflow-agent
Vector DB: d:\Github\research-workflow-agent\data\vector_db
Default topic: Key concepts for studying the Transformer architecture
HF token: configured


## 1. Load retriever

In [3]:
from rag.embeddings import ChunkEmbedder
from rag.retriever import DocumentRetriever
from rag.vectorstore import ChromaVectorStore

store = ChromaVectorStore(persist_directory=VECTOR_DB_PATH)
store.load_collection()

chunk_count = store.collection.count()
print(f"Collection: {store.collection_name} ({chunk_count} chunks)")

if chunk_count == 0:
    raise RuntimeError("Empty index. Run notebook 01_test_rag_pipeline (Part A) first.")

embedder = ChunkEmbedder()
retriever = DocumentRetriever(vector_store=store, embedder=embedder, top_k=TOP_K)

d:\Github\research-workflow-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: book_research (52 chunks)


## 2. Retrieve context (RAG)

In [4]:
context = retriever.invoke(QUESTION)

print(f"Topic: {QUESTION}")
print(f"Chunks retrieved: {len(context)}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7289.15it/s]


Topic: Key concepts for studying the Transformer architecture
Chunks retrieved: 5


## 3. Run Flashcard Agent

In [5]:
from agents.flashcard_agent import FlashcardAgent

print("Calling Flashcard Agent (Hugging Face API)...")
agent = FlashcardAgent()
result = agent.run(QUESTION, context)

Calling Flashcard Agent (Hugging Face API)...


In [6]:
print("Structured JSON output:")
print(result.to_json())

Structured JSON output:
{
  "flashcards": [
    {
      "front": "What is the Transformer architecture?",
      "back": "A model architecture that relies entirely on self-attention to draw global dependencies between input and output, eschewing recurrence."
    },
    {
      "front": "What is self-attention?",
      "back": "A mechanism that allows each position in a sequence to attend to all positions in the sequence, computing weighted sums of the input."
    },
    {
      "front": "What is multi-head attention?",
      "back": "A variant of self-attention that uses multiple attention heads to jointly attend to information from different representation subspaces."
    },
    {
      "front": "What is the encoder-decoder structure?",
      "back": "A common architecture for sequence transduction models, where the encoder maps input sequences to continuous representations and the decoder generates output sequences."
    },
    {
      "front": "What is residual connection?",
      "b

In [7]:
print("=" * 60)
print("FLASHCARDS")
print("=" * 60)

for i, card in enumerate(result.flashcards, start=1):
    print(f"\nCard {i}")
    print(f"  FRONT: {card.front}")
    print(f"  BACK:  {card.back}")

FLASHCARDS

Card 1
  FRONT: What is the Transformer architecture?
  BACK:  A model architecture that relies entirely on self-attention to draw global dependencies between input and output, eschewing recurrence.

Card 2
  FRONT: What is self-attention?
  BACK:  A mechanism that allows each position in a sequence to attend to all positions in the sequence, computing weighted sums of the input.

Card 3
  FRONT: What is multi-head attention?
  BACK:  A variant of self-attention that uses multiple attention heads to jointly attend to information from different representation subspaces.

Card 4
  FRONT: What is the encoder-decoder structure?
  BACK:  A common architecture for sequence transduction models, where the encoder maps input sequences to continuous representations and the decoder generates output sequences.

Card 5
  FRONT: What is residual connection?
  BACK:  A technique that adds the output of a sub-layer to its input, followed by layer normalization, to facilitate training and i

## 4. Test another topic

Change `another_topic` and run the cells below.

In [8]:
another_topic = "Attention mechanisms in neural networks"

context2 = retriever.invoke(another_topic)
result2 = agent.run(another_topic, context2)

print(json.dumps(result2.to_dict(), indent=2, ensure_ascii=False))

{
  "flashcards": [
    {
      "front": "What is self-attention?",
      "back": "An attention mechanism relating different positions of a single sequence to compute a representation of the sequence."
    },
    {
      "front": "What is the Transformer architecture?",
      "back": "A model architecture that relies entirely on an attention mechanism to draw global dependencies between input and output, eschewing recurrence."
    },
    {
      "front": "What is the main advantage of attention mechanisms?",
      "back": "Allowing modeling of dependencies without regard to their distance in the input or output sequences."
    },
    {
      "front": "What is Multi-Head Attention?",
      "back": "A technique used to counteract the reduced effective resolution due to averaging attention-weighted positions in the Transformer."
    },
    {
      "front": "What is the main challenge of sequential computation in neural networks?",
      "back": "The fundamental constraint of sequential co